# Heart Disease Prediction Workflow
This notebook covers a full data science pipeline for disease prediction, including:
- Data Preprocessing and Wrangling
- Descriptive Statistics and Data Analytics
- Data Visualization
- Handling of Outliers (Detection & Removal)
- Data Normalization (Standardization)
- Model Training (3 Algorithms)
- Hyperparameter Tuning
- Comparison of Models


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


## 1. Data Preprocessing & Data Wrangling
Looking at the dataset structure, handling null values if any.


In [ ]:
# Load dataset
df = pd.read_csv("HeartDisease.csv")
display(df.head())


In [ ]:
# Checking for null values
print("Missing values in each column:\n", df.isnull().sum())

# Dropping null values (if any)
df.dropna(inplace=True)
print("\nShape of dataset after removing nulls:", df.shape)


## 2. Descriptive Statistics & Data Analytics
Understanding the basic statistical measures of our continuous features and checking our target distribution.


In [ ]:
display(df.describe())
display(df.info())

# Target Distribution
print("\nTarget Value Counts:")
print(df['target'].value_counts())


## 3. Data Visualization
Plotting a correlation heatmap and histograms.


In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.show()

# Distribution of Target
sns.countplot(x='target', data=df, palette='Set2')
plt.title("Distribution of Heart Disease (Target)")
plt.show()


## 4. Outliers Detection and Removal
We utilize boxplots to detect outliers in continuous variables visually, then apply the Z-score method to securely remove extreme outliers.


In [ ]:
continuous_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

plt.figure(figsize=(15, 8))
for i, col in enumerate(continuous_features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[col], color='lightblue')
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()


In [ ]:
# Outlier Removal using Z-Score (threshold = 3)
z_scores = np.abs(stats.zscore(df[continuous_features]))
df_clean = df[(z_scores < 3).all(axis=1)]

print(f"Original Dataset Shape: {df.shape}")
print(f"Cleaned Dataset Shape: {df_clean.shape}")
print(f"Removed {len(df) - len(df_clean)} outlier rows.")


## 5. Data Normalization and Splitting
Next, we separate target from features, one-hot encode categorical features, and standardize continuous features so distance-based models (like SVC) perform optimally.


In [ ]:
# Separating target and features
X = df_clean.drop("target", axis=1)
y = df_clean["target"].values

# One-Hot Encoding for Categorical Features
categorical_features = ["cp", "slope", "ca", "restecg", "thal"]
# Ensuring columns exist in X before get_dummies (in case they differ)
cat_cols = [col for col in categorical_features if col in X.columns]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Feature Standardization on Continuous Features
scaler = StandardScaler()
# Ensuring continuous columns exist
cont_cols = [col for col in continuous_features if col in X.columns]
X[cont_cols] = scaler.fit_transform(X[cont_cols])

display(X.head())

# Splitting into Training and Test Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training Data Shape:", X_train.shape)
print("Test Data Shape:", X_test.shape)


## 6. Model Training (3 Algorithms) & 7. Hyperparameter Tuning
We will implement Logistic Regression, Random Forest, and Support Vector Classifier (SVC). 
We also apply GridSearchCV for hyperparameter tuning.


In [ ]:
# 1. Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

# 2. Random Forest with Hyperparameter Tuning
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_
rf_pred = rf_best.predict(X_test)
print("Best Random Forest Parameters:", rf_grid.best_params_)

# 3. SVC with Hyperparameter Tuning
svc_params = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}
svc_grid = GridSearchCV(SVC(random_state=42), svc_params, cv=5, scoring='accuracy', n_jobs=-1)
svc_grid.fit(X_train, y_train)
svc_best = svc_grid.best_estimator_
svc_pred = svc_best.predict(X_test)
print("Best SVC Parameters:", svc_grid.best_params_)


## 8. Comparison of Models
We compare the tuned models based on Accuracy, Precision, Recall, and F1-Score.


In [ ]:
def evaluate_model(name, y_true, y_preds):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_preds),
        "Precision": precision_score(y_true, y_preds),
        "Recall": recall_score(y_true, y_preds),
        "F1-Score": f1_score(y_true, y_preds)
    }

results = []
results.append(evaluate_model("Logistic Regression", y_test, lr_pred))
results.append(evaluate_model("Random Forest (Tuned)", y_test, rf_pred))
results.append(evaluate_model("SVC (Tuned)", y_test, svc_pred))

results_df = pd.DataFrame(results)
display(results_df)

# Plotting Comparison
results_df.set_index("Model").plot(kind='bar', figsize=(10, 6), colormap='viridis')
plt.title("Model Comparison Metrics")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.show()
